### Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm



### Generating synthetic data

In [ ]:
rng = np.random.default_rng(4)
n = 80
sigma = 0.12

x = rng.uniform(-2.0, 2.0, size=n)
y = x**2 + rng.normal(0.0, sigma, size=n)

df_demo = pd.DataFrame({"x": x, "y": y})
df_demo.head()


#### Fit OLS and test $(H_0: \beta_1 = 0)$

Our null hypothesis says that there is no linear relationship between the predictor and the response variables

We use **statsmodels** so we get classical **standard errors** and a **two-sided** \(p\)-value for the slope.

Use $\alpha$ = 0.05 --> if p-value < 0.05, reject $H_0$


In [ ]:
X_ols = sm.add_constant(df_demo["x"], has_constant="add")
y_ols = df_demo["y"].to_numpy(dtype=np.float64)

ols = sm.OLS(y_ols, X_ols).fit()
print(ols.summary().tables[1])

p_slope = float(ols.pvalues.loc["x"])
b1 = float(ols.params.loc["x"])
print(f"\nEstimated slope: {b1:.4f}")
print(f"Two-sided p-value for H0: beta_1 = 0: {p_slope:.6g}")


According to our p-value, we reject $H_0: \beta_1 = 0$ 

### Plotting least squares approximation against observations

In [ ]:
xgrid = np.linspace(df_demo["x"].min(), df_demo["x"].max(), 200)
Xg = sm.add_constant(xgrid, has_constant="add")
yhat_grid = ols.predict(Xg)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df_demo["x"], df_demo["y"], alpha=0.75, label="Observed data")
ax.plot(xgrid, yhat_grid, color="darkorange", lw=2.5, label="OLS fitted line")
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.legend()
plt.tight_layout()
plt.show()


### Observations

What do you observe about the shape of the observed data? Any chance that OLS regression may have been a mistake? What assumptions could we have violated?

Hmm... maybe we should check our assumptions

### Checking Assumptions
Now, let's check our suspicions. Maybe it's true that we shouldn't have used OLS regression to begin with!

We will test our assumptions using the following plots and summaries:

1. Residual distribution (skewedness and shape): Residual vs. Gaussian histogram
2. Homoscedasticity and linearity: Residual vs. fitted plot
3. Residual distribution (normality and independence of residuals): Normal Q–Q plot
4. **Global fit:** $R^2$ and **adjusted $R^2$** for the **linear** model — what fraction of the variance in $y$ the straight line explains on this sample (often modest when the true mean is curved). We also print $R^2$ for a **quadratic** model including $x^2$ as a contrast.

In [ ]:
fitted = ols.fittedvalues
resid = ols.resid



m_lin = smf.ols("y ~ x", data=df_demo).fit()
m_quad = smf.ols("y ~ x + I(x**2)", data=df_demo).fit()
anova_tbl = anova_lm(m_lin, m_quad)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(resid, bins=20, density=True, alpha=0.75, color="steelblue", edgecolor="white")
xs = np.linspace(resid.min(), resid.max(), 200)
axes[0].plot(
    xs,
    stats.norm.pdf(xs, loc=resid.mean(), scale=resid.std(ddof=1)),
    "r-",
    lw=2,
    label="Gaussian (sample μ, σ)",
)
axes[0].set_xlabel("Residual")
axes[0].set_ylabel("Density")
axes[0].set_title("Residual histogram vs Gaussian")
axes[0].legend(fontsize=8)

axes[1].scatter(fitted, resid, alpha=0.7, edgecolors="none")
axes[1].axhline(0, color="k", lw=1)
axes[1].set_xlabel("Fitted value")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs fitted (U-shape ⇒ linearity doubtful)")

stats.probplot(resid, dist="norm", plot=axes[2])
axes[2].set_title("Normal Q–Q (residuals)")
axes[2].set_xlabel("Theoretical quantiles")
axes[2].set_ylabel("Ordered residuals")

plt.tight_layout()
plt.show()

print(f"Linear model R²: {ols.rsquared:.4f}")
print(f"Linear model adjusted R²: {ols.rsquared_adj:.4f}")
print(f"Quadratic model R² (y ~ x + x², contrast): {m_quad.rsquared:.4f}")


print("\nLinear vs quadratic (nested model comparison):")
print(anova_tbl.to_string())


### Part 2 — Ticketing dataset (`train.csv` / `train2.csv`)

### Process data

In [ ]:
train_df = pd.read_csv("train.csv")
print(train_df.head(5))

target = "tickets_sold_first_hour"
X = train_df.drop(columns=[target])
y = train_df[target]

# split up training set into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("Training dataset size: ", X_train.shape)
print("Validation dataset size: ", X_val.shape)

### Data exploration

In [ ]:
print("Feature scales:\n")

for col in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[col]):
        print(
            f"{col}: min={X_train[col].min()}, max={X_train[col].max()}, "
            f"mean={X_train[col].mean():.2f}, std={X_train[col].std():.2f}"
        )
    else:
        print(f"{col}: categories={sorted(X_train[col].unique())}")

### Assumption-oriented diagnostics on train.csv

We assess **linearity and homoscedasticity** with a **residuals vs fitted** plot and **normality of residuals** with a **histogram** (compared to a Gaussian with the residual sample mean and standard deviation) and a **normal Q–Q plot** (visual checks only—no formal tests). **R²**, **RMSE**, and **MAE** summarize predictive fit on train/validation splits.



In [ ]:
cat_cols = ["seat_type", "speed_category"]


def encode(X):
    return pd.get_dummies(X, columns=cat_cols, drop_first=True)


X_train_enc = encode(X_train)
X_val_enc = encode(X_val)
X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)

model = LinearRegression()
model.fit(X_train_enc, y_train)

pred_train = model.predict(X_train_enc)
pred_val = model.predict(X_val_enc)

res_val = y_val.to_numpy() - pred_val


def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5


print("--- train.csv: prediction quality ---")
print(f"Train  R²:    {r2_score(y_train, pred_train):.4f}")
print(f"Val    R²:    {r2_score(y_val, pred_val):.4f}")
print(f"Train  RMSE:  {rmse(y_train, pred_train):.3f}")
print(f"Val    RMSE:  {rmse(y_val, pred_val):.3f}")
print(f"Train  MAE:   {mean_absolute_error(y_train, pred_train):.3f}")
print(f"Val    MAE:   {mean_absolute_error(y_val, pred_val):.3f}")

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].scatter(pred_val, res_val, alpha=0.25, s=10, edgecolors="none")
ax[0].axhline(0, color="k", lw=1)
ax[0].set_xlabel("Predicted (fitted), validation")
ax[0].set_ylabel("Residual")
ax[0].set_title("Residuals vs fitted (validation)")

ax[1].hist(res_val, bins=60, density=True, alpha=0.75, color="steelblue", edgecolor="white")
xs = np.linspace(res_val.min(), res_val.max(), 200)
ax[1].plot(
    xs,
    stats.norm.pdf(xs, loc=res_val.mean(), scale=res_val.std(ddof=1)),
    "r-",
    lw=2,
    label="Gaussian (μ̂, σ̂ from residuals)",
)
ax[1].set_xlabel("Residual")
ax[1].set_ylabel("Density")
ax[1].set_title("Residual histogram vs Gaussian")
ax[1].legend(fontsize=8)

stats.probplot(res_val, dist="norm", plot=ax[2])
ax[2].set_title("Normal Q–Q (validation residuals)")
plt.tight_layout()
plt.show()


### Testing assumptions on train2.csv

In [ ]:
target = "tickets_sold_first_hour"


def lr_val_metrics(csv_path, label, random_state=42):
    df = pd.read_csv(csv_path)
    print(f"{label}: {len(df):,} rows × {df.shape[1]} columns (incl. target)")
    X = df.drop(columns=[target])
    y = df[target]
    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )
    X_tr_e = encode(X_tr)
    X_va_e = encode(X_va).reindex(columns=X_tr_e.columns, fill_value=0)
    m = LinearRegression().fit(X_tr_e, y_tr)
    pred_tr = m.predict(X_tr_e)
    pred_va = m.predict(X_va_e)
    res_va = y_va.to_numpy() - pred_va

    def rmse_(a, b):
        return mean_squared_error(a, b) ** 0.5

    print(f"\n--- {label}: prediction quality ---")
    print(f"Train  R²:    {r2_score(y_tr, pred_tr):.4f}")
    print(f"Val    R²:    {r2_score(y_va, pred_va):.4f}")
    print(f"Train  RMSE:  {rmse_(y_tr, pred_tr):.3f}")
    print(f"Val    RMSE:  {rmse_(y_va, pred_va):.3f}")
    print(f"Train  MAE:   {mean_absolute_error(y_tr, pred_tr):.3f}")
    print(f"Val    MAE:   {mean_absolute_error(y_va, pred_va):.3f}")

    return {
        "label": label,
        "pred_va": pred_va,
        "res_va": res_va,
    }


out_train2 = lr_val_metrics("train2.csv", "train2.csv")


In [ ]:
def diagnostic_row(out, row_ax, rng_seed):
    rng = np.random.default_rng(rng_seed)
    pred_va = out["pred_va"]
    res_va = out["res_va"]
    n_show = min(8000, len(res_va))
    ix = rng.choice(len(res_va), size=n_show, replace=False)
    p_s, r_s = pred_va[ix], res_va[ix]

    row_ax[0].scatter(p_s, r_s, alpha=0.25, s=12, edgecolors="none")
    row_ax[0].axhline(0, color="k", lw=1)
    row_ax[0].set_ylabel(out["label"] + " — residual")
    row_ax[0].set_title("Residuals vs fitted (val subsample)")

    row_ax[1].hist(res_va, bins=60, density=True, alpha=0.75, color="steelblue", edgecolor="white")
    xs = np.linspace(res_va.min(), res_va.max(), 200)
    row_ax[1].plot(
        xs,
        stats.norm.pdf(xs, loc=res_va.mean(), scale=res_va.std(ddof=1)),
        "r-",
        lw=2,
        label="Gaussian (μ̂, σ̂)",
    )
    row_ax[1].set_ylabel("Density")
    row_ax[1].set_title("Residual histogram vs Gaussian")
    row_ax[1].legend(fontsize=8)

    stats.probplot(res_va, dist="norm", plot=row_ax[2])
    row_ax[2].set_title("Normal Q–Q (validation residuals)")


fig, axes = plt.subplots(1, 3, figsize=(14, 4))
diagnostic_row(out_train2, axes, rng_seed=2)
axes[0].set_xlabel("Predicted (fitted)")
axes[1].set_xlabel("Residual")
axes[2].set_xlabel("Theoretical quantiles")
plt.tight_layout()
plt.show()


### Food for thought

1. How would you describe the data from train2.csv? Is it skewed, what is its shape?
2. Even if you don't know of any other linear models, consider how you would describe the data (e.g., quadratic, logarithmic, etc)
3. If we were to fit train2.csv's data to an OLS regression and calculate p-values, would they be trustworthy? Why or why not?